# Audio Insight — VibeVoice-ASR Transcription Service

This notebook runs VibeVoice-ASR on a Colab GPU and exposes it as an API
that the local FastAPI backend can call for transcription.

**How it works:**
1. The model loads onto the Colab GPU
2. A Flask server starts with a `/transcribe` endpoint
3. ngrok creates a public URL local backend can reach it
4. backend sends audio URLs → Colab transcribes → returns results

---
## 1. Install dependencies

Installs the custom Transformers fork with VibeVoice support,
plus Flask for the API server and pyngrok for the public tunnel.

In [1]:
!pip install -q git+https://github.com/ebezzam/transformers.git@vibevoice_asr
!pip install -q torch torchaudio accelerate flask pyngrok
!pip install -q --upgrade yt-dlp

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


---
## 2. Verify GPU

Confirms a GPU is available. If not, change the runtime type above.

In [2]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected")

GPU: NVIDIA L4
VRAM: 23.7 GB


---
## 3. Load the model

Downloads and loads VibeVoice-ASR (~16GB).
First run takes 2-3 minutes for the download, then ~1 minute to load into GPU memory.

In [3]:
import time
import torch
from transformers import AutoProcessor, VibeVoiceAsrForConditionalGeneration

MODEL_ID = "microsoft/VibeVoice-ASR-HF"

torch.cuda.empty_cache()

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model...")
start = time.time()
model = VibeVoiceAsrForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
print(f"Model loaded in {time.time() - start:.1f}s")
print(f"Device: {model.device} | Dtype: {model.dtype}")
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

Loading processor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading model...


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/901 [00:00<?, ?it/s]

Model loaded in 7.4s
Device: cuda:0 | Dtype: torch.float16
Free VRAM: 6.8 GB


---
## 4. Quick test

Transcribes a short sample audio to verify the model works.
This also warms up the model — subsequent transcriptions will be faster.

In [4]:
TEST_AUDIO = "https://huggingface.co/datasets/bezzam/vibevoice_samples/resolve/main/realtime_model/vibevoice_tts_german.wav"

print("Transcribing test audio...")
start = time.time()

# Prepare audio for the model
inputs = processor.apply_transcription_request(
    audio=TEST_AUDIO,
).to(model.device, model.dtype)

# Run the model — generates token IDs
output_ids = model.generate(**inputs)

# Remove input tokens, keep only the generated transcription
generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

# Decode into structured segments: {Start, End, Speaker, Content}
result = processor.decode(generated_ids, return_format="parsed")[0]

print(f"Done in {time.time() - start:.1f}s\n")
for segment in result:
    print(segment)

Transcribing test audio...


Both `max_new_tokens` (=32768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Done in 6.1s

{'Start': 0.0, 'End': 7.56, 'Speaker': 0, 'Content': 'Revevoices is a novel framework designed for generating expressive, long-form, multi-speaker conversational audio.'}


---
## 5. Configure ngrok

ngrok creates a public URL that tunnels to the Flask server running inside Colab.
This is how backend can reach the transcription API.

Get your free auth token from: https://dashboard.ngrok.com/get-started/your-authtoken

In [5]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3BpFVAe21xcK8ckajbDZg3fCfHx_2N2i4W3ttigfU4U4sKRWK"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

---
## 6. Start the transcription API

Starts a Flask server with two endpoints:
- `GET /health` — check if the server is running
- `POST /transcribe` — send an audio URL, get back transcription

The server runs in a background thread so the cell finishes.
ngrok then creates the public URL.

**Copy the URL printed below and update your backend config.**

**Keep this notebook running while using the extension!**

In [6]:
import os
import uuid
import subprocess
from flask import Flask, request, jsonify
from threading import Thread

app = Flask(__name__)

DOWNLOAD_DIR = "/content/audio_downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)


def download_audio_from_url(url: str) -> str:
    """
    Downloads audio from a YouTube URL using yt-dlp.
    Returns the path to the downloaded wav file.
    """
    file_id = uuid.uuid4().hex[:10]
    output_path = os.path.join(DOWNLOAD_DIR, f"{file_id}")

    subprocess.run([
        "yt-dlp",
        "--extract-audio",
        "--audio-format", "wav",
        "--output", f"{output_path}.%(ext)s",
        "--no-playlist",
        "--quiet",
        url,
    ], check=True)

    return f"{output_path}.wav"


def transcribe_audio(audio_path: str, context: str = None) -> dict:
    """
    Core transcription function.
    Takes a local audio file path and optional context, returns results.
    """
    kwargs = {"audio": audio_path}
    if context:
        kwargs["prompt"] = context

    inputs = processor.apply_transcription_request(
        **kwargs
    ).to(model.device, model.dtype)

    output_ids = model.generate(**inputs)
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    parsed = processor.decode(generated_ids, return_format="parsed")[0]
    full_text = processor.decode(generated_ids, return_format="transcription_only")[0]

    return {"segments": parsed, "full_text": full_text}


# ---------- Routes ----------

@app.route("/health", methods=["GET"])
def health():
    """Health check — verifies the server and model are running."""
    return jsonify({"status": "ok", "model": "VibeVoice-ASR"})


@app.route("/transcribe", methods=["POST"])
def transcribe():
    """
    Transcription endpoint.
    Expects JSON: {"audio_url": "...", "context": "..." (optional)}
    The audio_url can be a YouTube URL or a direct audio file URL.
    Returns JSON: {"status", "segments", "full_text", "processing_time"}
    """
    audio_path = None
    try:
        data = request.get_json()
        audio_url = data.get("audio_url")
        context = data.get("context", None)

        if not audio_url:
            return jsonify({"error": "audio_url is required"}), 400

        start = time.time()

        # Check if it's a YouTube URL — if so, download first
        if "youtube.com" in audio_url or "youtu.be" in audio_url:
            print(f"Downloading from YouTube: {audio_url}")
            audio_path = download_audio_from_url(audio_url)
            print(f"Downloaded to: {audio_path}")
        else:
            # Direct audio URL — pass it straight to the model
            audio_path = audio_url

        print(f"Transcribing: {audio_path}")
        result = transcribe_audio(audio_path, context)

        elapsed = time.time() - start
        print(f"Done in {elapsed:.1f}s")

        return jsonify({
            "status": "success",
            "segments": result["segments"],
            "full_text": result["full_text"],
            "processing_time": round(elapsed, 2),
        })

    except subprocess.CalledProcessError as e:
        print(f"Download error: {str(e)}")
        return jsonify({"error": "Failed to download audio from URL"}), 400

    except Exception as e:
        print(f"Error: {str(e)}")
        return jsonify({"error": str(e)}), 500

    finally:
        # Clean up downloaded file to save disk space
        if audio_path and os.path.exists(audio_path):
            os.remove(audio_path)


# ---------- Start server + tunnel ----------

PORT = 5000

thread = Thread(target=lambda: app.run(port=PORT))
thread.daemon = True
thread.start()

public_url = ngrok.connect(PORT)

print("\n" + "=" * 60)
print("Transcription API is live!")
print(f"\n   URL:        {public_url}")
print(f"   Health:     {public_url}/health")
print(f"   Transcribe: {public_url}/transcribe  (POST)")
print("\n" + "=" * 60)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



Transcription API is live!

   URL:        NgrokTunnel: "https://untransported-unfeasibly-dora.ngrok-free.dev" -> "http://localhost:5000"
   Health:     NgrokTunnel: "https://untransported-unfeasibly-dora.ngrok-free.dev" -> "http://localhost:5000"/health
   Transcribe: NgrokTunnel: "https://untransported-unfeasibly-dora.ngrok-free.dev" -> "http://localhost:5000"/transcribe  (POST)



---
## 7. Test the API (optional)

Calls the API endpoints to verify everything works through the ngrok tunnel.

In [7]:
import requests

# Extract the URL string from the ngrok tunnel object
base_url = public_url.public_url if hasattr(public_url, 'public_url') else str(public_url)

# --- Test health ---
print("Testing /health...")
response = requests.get(f"{base_url}/health")
print(f"Response: {response.json()}\n")

# --- Test transcription ---
print("Testing /transcribe...")
response = requests.post(
    f"{base_url}/transcribe",
    json={
        "audio_url": "https://huggingface.co/datasets/bezzam/vibevoice_samples/resolve/main/realtime_model/vibevoice_tts_german.wav",
        "context": "About VibeVoice",
    },
)
result = response.json()
print(f"Status: {result['status']}")
print(f"Time: {result['processing_time']}s")
print(f"Text: {result['full_text']}\n")
print("Segments:")
for seg in result["segments"]:
    speaker = seg.get('Speaker', '?')
    print(f"  [{seg['Start']:.1f}s - {seg['End']:.1f}s] Speaker {speaker}: {seg['Content']}")

Testing /health...


INFO:werkzeug:127.0.0.1 - - [12/Apr/2026 19:42:31] "GET /health HTTP/1.1" 200 -


Response: {'model': 'VibeVoice-ASR', 'status': 'ok'}

Testing /transcribe...
Transcribing: https://huggingface.co/datasets/bezzam/vibevoice_samples/resolve/main/realtime_model/vibevoice_tts_german.wav


Both `max_new_tokens` (=32768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [12/Apr/2026 19:42:37] "POST /transcribe HTTP/1.1" 200 -


Done in 5.3s
Status: success
Time: 5.3s
Text: VibeVoice is this novel framework designed for generating expressive, long-form, multi-speaker, conversational audio.

Segments:
  [0.0s - 7.6s] Speaker 0: VibeVoice is this novel framework designed for generating expressive, long-form, multi-speaker, conversational audio.
